# Q1 after the gas-market event

The same six profiles as `q1_where.ipynb`, over `table_h_price_post` instead of
`table_h_price_load`: hours from 2023-03-01 onward, 30,719 of the 49,655.

Built to answer one question: which of the shapes in the first notebook were the market,
and which were the crisis. Switch `TABLE` below to compare the two directly.

A scratchpad. Whatever survives is promoted into a script; outputs are stripped on commit.


In [ ]:
import duckdb
import matplotlib.pyplot as plt

# Set to "table_h_price_load" to re-run every cell over the whole range instead.
TABLE = "table_h_price_post"

con = duckdb.connect("../data/processed/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")


def q(sql):
    """Run SQL and return a pandas DataFrame, so matplotlib has a table to draw."""
    return con.execute(sql).df()


def profile(key):
    """The negative and extreme share of hours, grouped by one key."""
    return q(f"""
        SELECT {key},
               100.0 * COUNT(*) FILTER (WHERE price_neg)     / COUNT(*) AS neg_pct,
               100.0 * COUNT(*) FILTER (WHERE extreme_price) / COUNT(*) AS ext_pct
        FROM {TABLE} GROUP BY 1 ORDER BY 1
    """)


def grid(keys, measure):
    """A two-key profile, pivoted for a heatmap."""
    row, col = keys
    return q(f"""
        SELECT {row}, {col},
               100.0 * COUNT(*) FILTER (WHERE {measure}) / COUNT(*) AS pct
        FROM {TABLE} GROUP BY 1, 2
    """).pivot(index=row, columns=col, values="pct")


def line(df, xlabel, xticks=None):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(df.iloc[:, 0], df.neg_pct, marker="o", label="negative")
    ax.plot(df.iloc[:, 0], df.ext_pct, marker="o", label="extreme")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("% of hours in that group")
    if xticks is not None:
        ax.set_xticks(xticks)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    return fig, ax


def heat(g, xlabel, ylabel, clabel, xstep=1):
    """Tick labels come from the frame, so the axes cannot disagree with the data."""
    fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.imshow(g, aspect="auto", origin="lower")
    ax.set_xticks(range(0, len(g.columns), xstep), g.columns[::xstep])
    ax.set_yticks(range(len(g.index)), g.index)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.colorbar(im, ax=ax, label=clabel)
    fig.tight_layout()
    return fig, ax


print(
    TABLE,
    q(f"SELECT COUNT(*) AS hours, MIN(date_ams) AS f, MAX(date_ams) AS l FROM {TABLE}").to_string(
        index=False
    ),
)

## G_HOUR, G_DOW, G_MONTH, G_YEAR


In [ ]:
line(profile("hour_date"), "hour of day (UTC)", xticks=range(0, 24, 2))
line(profile("day_of_week"), "day of week")
line(profile("month_date"), "month", xticks=range(1, 13))
line(profile("year_date"), "year")

## G_HOUR_MONTH and G_MONTH_YEAR


In [ ]:
heat(
    grid(("month_date", "hour_date"), "price_neg"),
    "hour of day (UTC)",
    "month",
    "% negative",
    xstep=2,
)
heat(
    grid(("month_date", "hour_date"), "extreme_price"),
    "hour of day (UTC)",
    "month",
    "% extreme",
    xstep=2,
)
heat(grid(("year_date", "month_date"), "price_neg"), "month", "year", "% negative")
heat(grid(("year_date", "month_date"), "extreme_price"), "month", "year", "% extreme")

## What to check against the first notebook

- The negative peaks held in the whole-range version: hour 13, Sunday, May, May at 14:00.
- The extreme peaks did not: December collapsed from 23.79% to 1.84% of hours once the
  crisis years were out, and the hour-month peak moved from December 17:00 to September 19:00.
- `extreme_price` still uses the 225.00 bar from the whole range. Over this period that flags
  241 hours in 30,719, not a tenth of them.
